In [7]:
import numpy as np
from collections import defaultdict
from matchingMarkets import hri

# School capacities and preference weights
SCHOOL_CAPACITIES = [5, 40, 15, 10]
PREFERENCE_WEIGHTS = [5, 1, 2, 3]  # For schools 1, 2, 3, 4
NUM_STUDENTS = 60
NUM_SCHOOLS = 4
NUM_SIMULATIONS = 5000

# Priority structure
PRIORITIES = {
    0: [list(range(1, 16)), list(range(16, 31)), list(range(31, 46)), list(range(46, 61))],  # School 1
    1: [list(range(46, 61)), list(range(1, 16)), list(range(16, 31)), list(range(31, 46))],  # School 2
    2: [list(range(31, 46)), list(range(46, 61)), list(range(1, 16)), list(range(16, 31))],  # School 3
    3: [list(range(16, 31)), list(range(31, 46)), list(range(46, 61)), list(range(1, 16))]   # School 4
}

def generate_student_preferences():
    """Generate random student preferences using given weights."""
    preferences = np.zeros((NUM_STUDENTS, NUM_SCHOOLS), dtype=int)
    for i in range(NUM_STUDENTS):
        probs = np.array(PREFERENCE_WEIGHTS) / sum(PREFERENCE_WEIGHTS)
        pref = np.random.choice(NUM_SCHOOLS, size=NUM_SCHOOLS, replace=False, p=probs)
        preferences[i] = pref
    return preferences  # nStudents x nSchools

def single_tie_breaking():
    """Generate a single random lottery for all schools."""
    lottery = np.random.permutation(NUM_STUDENTS) + 1
    return {student: rank for rank, student in enumerate(lottery)}  # Student ID -> rank

def multiple_tie_breaking():
    """Generate separate random lotteries for each school."""
    tie_breakers = {}
    for school in range(NUM_SCHOOLS):
        lottery = np.random.permutation(NUM_STUDENTS) + 1
        tie_breakers[school] = {student: rank for rank, student in enumerate(lottery)}
    return tie_breakers

def create_strict_priorities(priorities, tie_breaker, single=True):
    """Create strict priority lists by breaking ties."""
    strict_priorities = []
    for school in range(NUM_SCHOOLS):
        school_priority = []
        for group in priorities[school]:
            if single:
                sorted_group = sorted(group, key=lambda x: tie_breaker[x])
            else:
                sorted_group = sorted(group, key=lambda x: tie_breaker[school][x])
            school_priority.extend(sorted_group)
        # Convert to 0-based indexing for matchingMarkets
        strict_priorities.append([s - 1 for s in school_priority])
    # Convert to c.prefs format: nSchools x nStudents
    return np.array(strict_priorities)  # Each row is a school's ranking of students

def run_simulation():
    """Run simulations for both tie-breaking rules."""
    single_results = defaultdict(int)
    multiple_results = defaultdict(int)

    for _ in range(NUM_SIMULATIONS):
        # Generate student preferences (0-based for matchingMarkets)
        s_prefs = generate_student_preferences()  # nStudents x nSchools

        # Single tie-breaking simulation
        single_tie_breaker = single_tie_breaking()
        c_prefs_single = create_strict_priorities(PRIORITIES, single_tie_breaker, single=True)
        # Run hri (expects 0-based indices)
        matching = hri(s_prefs=s_prefs, c_prefs=c_prefs_single, c_caps=SCHOOL_CAPACITIES)
        # matching is a list of length nStudents, where matching[i] is the school (0-based) or -1 if unassigned
        for student in range(NUM_STUDENTS):
            school = matching[student]
            if school != -1:
                pref_rank = np.where(s_prefs[student] == school)[0][0]
                single_results[pref_rank] += 1

        # Multiple tie-breaking simulation
        multiple_tie_breaker = multiple_tie_breaking()
        c_prefs_multiple = create_strict_priorities(PRIORITIES, multiple_tie_breaker, single=False)
        matching = hri(s_prefs=s_prefs, c_prefs=c_prefs_multiple, c_caps=SCHOOL_CAPACITIES)
        for student in range(NUM_STUDENTS):
            school = matching[student]
            if school != -1:
                pref_rank = np.where(s_prefs[student] == school)[0][0]
                multiple_results[pref_rank] += 1

    # Calculate averages
    single_averages = {i: single_results[i] / NUM_SIMULATIONS for i in range(NUM_SCHOOLS)}
    multiple_averages = {i: multiple_results[i] / NUM_SIMULATIONS for i in range(NUM_SCHOOLS)}
    
    return single_averages, multiple_averages

def main():
    single_averages, multiple_averages = run_simulation()
    
    print("Single Tie-Breaking Results (average number of students):")
    for i in range(NUM_SCHOOLS):
        print(f"Preference {i+1}: {single_averages[i]:.2f}")
    
    print("\nMultiple Tie-Breaking Results (average number of students):")
    for i in range(NUM_SCHOOLS):
        print(f"Preference {i+1}: {multiple_averages[i]:.2f}")
    
    # Answers to questions
    print("\nAnswers to Questions:")
    print("a) Practical Advantages:")
    print("- Single Tie-Breaking:")
    print("  - Simpler to implement and manage (one lottery vs multiple)")
    print("  - More transparent and easier to verify")
    print("  - Consistent ordering across schools")
    print("- Multiple Tie-Breaking:")
    print("  - Potentially fairer as students have independent chances at each school")
    print("  - Reduces impact of a single bad lottery draw")
    
    print("\nb) Recommendation:")
    first_choice_single = single_averages[0]
    first_choice_multiple = multiple_averages[0]
    last_choice_single = single_averages[3]
    last_choice_multiple = multiple_averages[3]
    
    print(f"First choice assignments - Single: {first_choice_single:.2f}, Multiple: {first_choice_multiple:.2f}")
    print(f"Last choice assignments - Single: {last_choice_single:.2f}, Multiple: {last_choice_multiple:.2f}")
    
    if first_choice_multiple > first_choice_single and last_choice_multiple < last_choice_single:
        print("Recommendation: Use multiple tie-breaking rule.")
        print("Reason: It assigns more students to their first-choice school and fewer to their least-preferred school, meeting the district's criteria.")
    elif first_choice_single > first_choice_multiple and last_choice_single < last_choice_multiple:
        print("Recommendation: Use single tie-breaking rule.")
        print("Reason: It assigns more students to their first-choice school and fewer to their least-preferred school, meeting the district's criteria.")
    else:
        print("Recommendation: Either method could be used.")
        print("Reason: The results are mixed. Consider other factors like implementation complexity (favoring single) or perceived fairness (favoring multiple).")

if __name__ == "__main__":
    np.random.seed(42)  # For reproducibility
    main()

ModuleNotFoundError: No module named 'matchingMarkets'